# 🧪 W4-D3 概念实验：混合检索、RRF、重排序、查询改写各救了什么？

> 配套阅读：`ima/第4周-Day3-高级RAG混合检索与重排序.md`（三种技术的完整原理与误区在那边）
>
> 基础向量检索有"盲区"。这个 notebook 用一个 8 条文档的迷你语料，把三个补丁逐个跑通：
> 1. **语义盲区 vs 符号盲区**：语义向量查不了 "SKU-207"，BM25 理解不了"冰爽解暑"——各瘸一条腿；
> 2. **RRF 融合**：两路排名怎么合成一路才公平？
> 3. **Multi-Query 改写 + 重排序**：词汇鸿沟怎么跨、粗排排错的怎么捞回来。
>
> 实验环境：纯 Python/numpy。语义向量用"极简词典向量"模拟 Embedding 的泛化能力（md 里有说明）。

## 准备：8 条糖水店文档 + 两种检索器

- **模拟语义向量**：用一个小词典把文本映射到 4 维主题向量 `[冰爽, 果味, 奶香, 解腻]`，
  模拟真实 Embedding"按意思而非按字匹配"的能力（词典 = 我们手工注入的"语义知识"）；
- **BM25**：字符 bigram + 标准打分公式（词频饱和 + 逆文档频率），擅长精确匹配罕见符号。

In [ ]:
import math
import numpy as np

DOCS = {
    "D1": "杨枝甘露：芒果与西柚果肉，椰浆打底，冰镇出品，夏季限定，售价20元。",
    "D2": "芒果双皮奶，产品编号SKU-207，双份奶皮，售价18元。",
    "D3": "双皮奶：顺德水牛奶隔水蒸制，奶香浓郁，常温出品，售价15元。",
    "D4": "芋泥波波冰：芋头现蒸捣泥，黑糖波波铺顶，冰沙绵密，售价17元。",
    "D5": "红豆沙：陈皮慢炖，冬季热饮，暖胃首选，售价12元。",
    "D6": "西瓜冰：纯西瓜果肉冰沙，无奶配方，夏天人气王，售价16元。",
    "D7": "桂花酸梅汤：古法熬煮，生津解腻，开胃消食，售价10元。",
    "D8": "会员卡充值300送50，全场饮品第二杯半价。",
}

# ---------- 检索器1：模拟语义向量（词典 Embedding） ----------
ICE    = ["冰镇", "冰沙", "夏季", "夏天", "冷饮", "冰爽", "解暑", "冰"]
FRUIT  = ["芒果", "西瓜", "西柚", "果肉", "水果"]
MILK   = ["奶", "牛奶", "双皮奶", "奶皮", "奶香", "炼乳", "椰浆", "乳"]
SOOTHE = ["解腻", "开胃", "生津", "消食", "酸梅"]
TOPICS = [ICE, FRUIT, MILK, SOOTHE]

def embed(text):
    v = np.zeros(4)
    for d, kws in enumerate(TOPICS):
        v[d] = sum(text.count(k) for k in kws)
    return v

DOC_EMB = {k: embed(t) for k, t in DOCS.items()}

def vec_search(query, k=3):
    q = embed(query)
    if np.linalg.norm(q) < 1e-9:
        return []   # 零向量：词典里没有查询的任何主题词 → 语义检索失明
    qn = q / np.linalg.norm(q)
    sims = {k_: float(qn @ (v / np.linalg.norm(v))) if np.linalg.norm(v) > 0 else 0.0
            for k_, v in DOC_EMB.items()}
    return [(d, s) for d, s in sorted(sims.items(), key=lambda kv: -kv[1]) if s > 0][:k]

# ---------- 检索器2：BM25（字符 bigram） ----------
def bigrams(t):
    t = "".join(ch for ch in t if ch not in "，。：、！？")
    return [t[i:i+2] for i in range(len(t) - 1)]

DOC_TOK = {k: bigrams(t) for k, t in DOCS.items()}
AVGDL = np.mean([len(t) for t in DOC_TOK.values()])
DF = {}
for toks in DOC_TOK.values():
    for g in set(toks):
        DF[g] = DF.get(g, 0) + 1

def bm25(query, k=3, k1=1.5, b=0.75):
    qg = [g for g in bigrams(query) if DF.get(g)]   # 语料里没出现的词不参与
    scores = {}
    for k_, toks in DOC_TOK.items():
        s, dl = 0.0, len(toks)
        for g in qg:
            tf = toks.count(g)
            if tf:
                idf = math.log((len(DOCS) - DF[g] + 0.5) / (DF[g] + 0.5) + 1)
                s += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / AVGDL))
        scores[k_] = s
    return [(d, float(s)) for d, s in sorted(scores.items(), key=lambda kv: -kv[1]) if s > 0][:k]

print("两种检索器就绪。语料：", ", ".join(f"{k}({v[:8]}…)" for k, v in list(DOCS.items())[:4]), "…")

## 实验 1：各自的盲区 —— 语义型问题 vs 符号型问题

同一个语料，两类问题各让一种检索器失明：

- **语义型**："夏天想喝点冰爽的水果甜品"——字面和文档对不上，BM25 几乎哑火，词典向量稳稳召回冰品；
- **符号型**："产品SKU-207的价格"——词典里根本没有这些字 → 零向量（等价于随机），
  BM25 靠罕见 bigram `SK/KU/U-/07` 精确命中 D2。

In [ ]:
GROUND_TRUTH = {
    "夏天想喝点冰爽的水果甜品": {"D1", "D4", "D6"},   # 三款冰品
    "产品SKU-207的价格": {"D2"},
}

for q, truth in GROUND_TRUTH.items():
    print(f"查询：「{q}」  相关文档：{sorted(truth)}")
    vr, br = vec_search(q, 3), bm25(q, 3)
    fmt = lambda r: [(k_, round(s, 3)) for k_, s in r] if r else "（无结果：检索器失明）"
    print(f"  语义向量 Top3 : {fmt(vr)}  命中 {len({k_ for k_, _ in vr} & truth)}/3")
    print(f"  BM25     Top3 : {fmt(br)}  命中 {len({k_ for k_, _ in br} & truth)}/3")
    print()

print("结论：语义型问题 BM25 只靠「夏天」一个 bigram 捞到 D6（1/3）；")
print("     符号型问题语义向量变零向量（模拟真实 Embedding 对未知符号的无力感）。")
print("     → 单一检索器都有结构性盲区，这就是 Hybrid Search 的动机。")

## 实验 2：RRF 融合 —— 不比分数比名次

两路检索器的**分数量纲完全不同**（余弦 ∈ [0,1]，BM25 ∈ [0, +∞)），直接加权平均等于瞎调参。
RRF（Reciprocal Rank Fusion）只看名次：`score(d) = Σ 1/(k + rank_i(d))`，k 通常取 60。

用"芒果冰沙多少钱"演示：语义路推荐水果冰饮、BM25 路认"芒果""冰沙"字面证据，融合后两者都认可的文档登顶。

In [ ]:
def rrf(rankings, k=60):
    """rankings: 若干个 [(doc, score), ...] 名次列表 → 融合排名"""
    score = {}
    for ranking in rankings:
        for r, (doc, _) in enumerate(ranking, start=1):
            score[doc] = score.get(doc, 0.0) + 1.0 / (k + r)
    return sorted(score.items(), key=lambda kv: -kv[1])

q = "芒果冰沙多少钱"
vr, br = vec_search(q, 8), bm25(q, 8)
rank_v = {d: r for r, (d, _) in enumerate(vr, 1)}
rank_b = {d: r for r, (d, _) in enumerate(br, 1)}

print(f"查询：「{q}」\n")
print(f"{'文档':<5}{'语义名次':>8}{'BM25名次':>9}{'RRF得分':>10}")
for d, s in rrf([vr, br])[:5]:
    rv, rb = rank_v.get(d, "-"), rank_b.get(d, "-")
    print(f"{d:<5}{rv:>10}{rb:>11}{s:>10.5f}")

print("\nD1(杨枝甘露,芒果+冰镇) 和 D6(西瓜冰,冰沙) 两路都靠前 → RRF 分最高；")
print("D2 只有字面'芒果'证据、语义排后 → 融合后让位。两路共识 > 单路高分。")

## 实验 3：Multi-Query 查询改写 —— 跨过词汇鸿沟

顾客问"**吃完火锅喝什么**"——查询里一个主题词都没有，词典向量 → 零向量，检索失明。
把查询改写成 3 个不同角度的变体（解腻饮品 / 开胃酸梅汤 / 冰镇解暑），
每个变体各自检索，再 RRF 融合——相关文档 D7 被 3 票捞回第一。

In [ ]:
original = "吃完火锅喝什么"          # 词典里没有"火锅" → 零向量
variants = ["解腻饮品", "开胃酸梅汤", "冰镇酸梅汤"]
TARGET = "D7"   # 桂花酸梅汤：唯一"解腻"产品

def top1(rankings_list):
    return rrf(rankings_list)[0][0]

single = vec_search(original, 8)
if single:
    print(f"原始查询：「{original}」→ Top1 = {single[0][0]}")
else:
    print(f"原始查询：「{original}」→ 语义向量 = 零向量，检索器完全失明，返回空 ✗")

print("\n改写变体各自检索（Top2）：")
var_rankings = []
for v in variants:
    r = vec_search(v, 8)
    var_rankings.append(r)
    print(f"  「{v}」→ {[d for d, _ in r[:2]]}")

fused = rrf(var_rankings)
print(f"\n三变体 RRF 融合 Top3：{[d for d, _ in fused[:3]]}")
print(f"目标 D7 融合后名次：第 {[d for d, _ in fused].index(TARGET) + 1} 名 → {'✓ 召回' if fused[0][0] == TARGET else '✗'}")

print("\n结论：查询改写用'多种问法'去撞语料的词汇空间，")
print("     单查询 0% 召回 → 多查询融合 100% 召回（这就是 Multi-Query 的价值，也有 3 倍检索成本）。")

## 实验 4：重排序（Rerank）—— 粗排的错，精排来救；但救不了没进圈的

查询"**有奶的冰品哪个便宜**"：语义粗排把"有奶但常温"的 D3/D2 和"冰但无奶"的 D4 排在前面，正确答案 D1（椰浆=奶，冰镇=冰）只排在第 5，掉出 Top-3。
用一个更精细的打分器（奶✓ + 冰✓ + 价格低✓）在粗排 Top-K 里重排 → D1 登顶。
**但**：如果粗排只取 Top-3，D1 根本没进候选圈，神仙重排也救不回 → "粗排保底召回，精排负责排序"。

In [ ]:
q = "有奶的冰品哪个便宜"

# 粗排：语义向量 Top-8（全部文档）
coarse = vec_search(q, 8)
print("粗排（语义向量）名次：", [d for d, _ in coarse])
print("正确答案 D1（椰浆=奶，冰镇=冰，唯一同时满足两条件）粗排第",
      [d for d, _ in coarse].index("D1") + 1, "名\n")

def rerank_score(doc_id):
    t = DOCS[doc_id]
    has_milk = any(k in t for k in MILK if k != "奶") or ("奶" in t and "无奶" not in t)  # 精排器懂得"无奶"是否定
    has_ice = any(k in t for k in ICE)
    cheap = any(p in t for p in ["12元", "15元", "16元", "17元", "18元", "20元"]) and "300" not in t
    return 2 * has_milk + 2 * has_ice + 1 * (cheap and has_milk and has_ice)

for topk in (3, 8):
    cand = [d for d, _ in coarse[:topk]]
    reranked = sorted(cand, key=lambda d: -rerank_score(d))
    hit = "✓" if reranked[0] == "D1" else "✗（D1 没进粗排候选，重排无力回天）"
    print(f"粗排 Top-{topk} + 重排 → Top1 = {reranked[0]} {hit}")

print("\n结论：重排序只能在粗排给定的候选内重排（recall 上限由粗排决定）；")
print("     粗排深度 K 是安全垫，重排序是天花板——两层都要调。")

## 实验 5：全家福 —— 各策略在两类问题上的 Recall@3

把前面的实测数字汇总成一张图：单一检索器各瘸一条腿，RRF 混合两条腿走路。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 用前面的检索器实测（重新计算，保证可复现）
q_sem, q_sym = "夏天想喝点冰爽的水果甜品", "产品SKU-207的价格"
truth = {"夏天想喝点冰爽的水果甜品": {"D1", "D4", "D6"}, "产品SKU-207的价格": {"D2"}}

def recall3(method, q):
    got = {d for d, _ in method(q, 3)}
    return len(got & truth[q]) / len(truth[q])

def hybrid(q, k=3):
    fused = rrf([vec_search(q, 8), bm25(q, 8)])
    return fused[:k]

data = np.array([
    [recall3(vec_search, q_sem), recall3(vec_search, q_sym)],
    [recall3(bm25, q_sem),       recall3(bm25, q_sym)],
    [recall3(hybrid, q_sem),     recall3(hybrid, q_sym)],
])
methods = ["语义向量", "BM25", "RRF混合"]
xt = ["语义型问题\n(冰爽水果甜品)", "符号型问题\n(SKU-207)"]

x = np.arange(2)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
w = 0.25
for i, m in enumerate(methods):
    bars = ax.bar(x + (i - 1) * w, data[i], w, label=m)
    for b, v in zip(bars, data[i]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.0%}", ha="center", fontsize=9)
ax.set_xticks(x, xt)
ax.set_ylabel("Recall@3（本notebook实测）")
ax.set_ylim(0, 1.15)
ax.set_title("单检索器各瘸一条腿，RRF 混合检索两边都接住")
ax.legend()
plt.tight_layout()
plt.show()

print("复盘：混合检索的收益 = 语义泛化 ∪ 精确匹配；成本 = 两套索引 + 融合开销。")
print("     再叠加 Multi-Query（实验3）与 Rerank（实验4）就是完整的高级 RAG 检索栈。")

## 结论

| 技术 | 解决的盲区 | 实验证据 |
|---|---|---|
| Hybrid 混合检索 | 语义向量不识符号 / BM25 不懂语义 | 实验1：SKU-207 与"冰爽甜品"各让一路失明 |
| RRF 融合 | 分数量纲不可比 | 实验2：只比名次，两路共识者登顶 |
| Multi-Query 改写 | 查询与语料的词汇鸿沟 | 实验3：零向量查询 0% → 融合后 Top1 命中 |
| Rerank 重排序 | 粗排排错 | 实验4：D1 从第5到第1；Top-3 截断则救不回 |

→ 深入阅读：`ima/第4周-Day3-高级RAG混合检索与重排序.md`（HyDE、Cross-Encoder 原理、糖水店完整管道、4 个误区）